# 01 – EDA & Schema Discovery (CASMI 2026 / Enveda)

### Purpose
Perform end-to-end exploratory data analysis on the Enveda CASMI 2026 dataset:
- Auto-detect the schema with a robust `SchemaAdapter` handling nested arrays and JSON strings.
- Profile spectral properties and evaluate train-vs-test distribution shifts via Kolmogorov-Smirnov tests.
- High-performance molecule profiling using `joblib` multiprocessing across all 4 vCPUs and native tuple iterations.
- Dual NVIDIA T4 GPU tensor acceleration for neutral-mass overlap search and candidate ranking.
- Validate the adduct neutral mass conversion table against exact SMILES mass (sanity check).
- Demonstrate the exact MRR@25 competition metric (tautomer canonicalization, InChIKey14 deduplication).
- Save clean normalized parquet tables and metadata artifacts.

| Specification | Details |
| :--- | :--- |
| **Inputs** | `train.parquet` (or `train_head100.parquet`), `test.parquet` (or `test_head100.parquet`) |
| **Outputs** | `schema.json`, `train_clean.parquet`, `test_clean.parquet`, `stats.json`, `manifest.json`, `recommendations.md`, 6 figures |
| **Hardware Optimization** | 4 vCPU Multiprocessing (`joblib`) + Dual NVIDIA T4 GPU tensor sharding (`cuda:0`, `cuda:1`) |
| **Expected Runtime** | ~2–4 minutes on full dataset (~5 seconds on head100) |

In [ ]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================
from dataclasses import dataclass, field, asdict
import time
import os

@dataclass
class CFG:
    seed: int = 42
    SMOKE_TEST: bool = False
    smoke_n_rows: int = 500
    time_budget_s: int = 3600  # 1 hour max budget

    # Concurrency & Acceleration
    n_cpu_workers: int = 4           # 4 vCPUs on Kaggle instances
    mp_chunk_size: int = 250         # Batch size for worker processes
    use_gpu_acceleration: bool = True # Shard tensor search across Dual T4 GPUs

    # Path configuration: Primary Kaggle paths with fallbacks
    data_dir: str = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"
    output_dir: str = "/kaggle/working/artifacts/eda"

    # Preprocessing thresholds
    ppm_tolerance: float = 10.0      # Adduct neutral mass sanity check tolerance
    mass_match_ppm: float = 5.0      # Overlap analysis ppm threshold
    top_k_peaks: int = 200           # Cap for spectral peaks
    truncate_array_len: int = 8      # Display truncation length

cfg = CFG()
T0 = time.time()
print(f"[Config] SMOKE_TEST={cfg.SMOKE_TEST}, Seed={cfg.seed}, Workers={cfg.n_cpu_workers}, OutputDir={cfg.output_dir}")

In [ ]:
# ==============================================================================
# IMPORTS & ENVIRONMENT REPORT
# ==============================================================================
import sys
import gc
import json
import warnings
import random
import platform
from glob import glob
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import scipy
from scipy import stats as sp_stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)

# Mandatory RDKit check
try:
    import rdkit
    from rdkit import Chem, RDLogger
    from rdkit.Chem import Descriptors, AllChem, MolToSmiles, MolFromSmiles
    from rdkit.Chem import rdMolDescriptors, Scaffolds
    from rdkit.Chem.Scaffolds import MurckoScaffold
    from rdkit.Chem.MolStandardize import rdMolStandardize
    from rdkit.Chem.inchi import MolToInchi, InchiToInchiKey
    RDLogger.DisableLog("rdApp.*")
    RDKIT_VERSION = rdkit.__version__
except ImportError as err:
    raise ImportError(f"RDKit is mandatory for this notebook: {err}")

# PyTorch & Dual T4 GPU detection
try:
    import torch
    TORCH_VERSION = torch.__version__
    CUDA_AVAILABLE = torch.cuda.is_available()
    GPU_COUNT = torch.cuda.device_count() if CUDA_AVAILABLE else 0
    if CUDA_AVAILABLE:
        gpu_info = [f"GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)"
                    for i in range(GPU_COUNT)]
    else:
        gpu_info = ["No GPU detected (CPU mode)"]
except ImportError:
    TORCH_VERSION = "N/A"
    CUDA_AVAILABLE = False
    GPU_COUNT = 0
    gpu_info = ["PyTorch not installed"]

print(f"Python: {platform.python_version()} on {platform.system()} ({platform.machine()})")
print(f"CPU Cores Available: {os.cpu_count()} (Using {cfg.n_cpu_workers} workers)")
print(f"Libraries: numpy={np.__version__}, pandas={pd.__version__}, scipy={scipy.__version__}, rdkit={RDKIT_VERSION}")
print(f"Torch: {TORCH_VERSION} | CUDA: {CUDA_AVAILABLE} (Count: {GPU_COUNT}) | Devices: {'; '.join(gpu_info)}")

In [ ]:
# ==============================================================================
# UTILITIES (Seeding, Timers, Canonicalization, MRR@25 Scorer, Adducts)
# ==============================================================================

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except (ImportError, NameError):
        pass

seed_everything(cfg.seed)

def elapsed() -> float:
    return time.time() - T0

def check_budget():
    if elapsed() > cfg.time_budget_s:
        raise TimeoutError(f"Time budget of {cfg.time_budget_s}s exceeded (elapsed: {elapsed():.1f}s)")

# Tautomer canonicalization with lazy initialization (avoids multiprocessing pickling errors)
_tautomer_enumerator = None

def get_tautomer_enumerator():
    global _tautomer_enumerator
    if _tautomer_enumerator is None:
        _tautomer_enumerator = rdMolStandardize.TautomerEnumerator()
    return _tautomer_enumerator

_ik14_cache = {}

def ik14(smiles: str) -> str:
    """Compute InChIKey14 via RDKit tautomer canonicalization with caching."""
    if not smiles or not isinstance(smiles, str):
        return None
    if smiles in _ik14_cache:
        return _ik14_cache[smiles]
    try:
        mol = MolFromSmiles(smiles)
        if mol is None:
            _ik14_cache[smiles] = None
            return None
        enum = get_tautomer_enumerator()
        can_mol = enum.Canonicalize(mol)
        inchi = MolToInchi(can_mol)
        if not inchi:
            _ik14_cache[smiles] = None
            return None
        full_key = InchiToInchiKey(inchi)
        key14 = full_key[:14] if full_key else None
    except Exception:
        key14 = None
    _ik14_cache[smiles] = key14
    return key14

def compute_mrr25(predictions: dict, ground_truth: dict) -> float:
    """
    Compute competition metric MRR@25.
    - predictions: {mol_id: [smiles1, smiles2, ...]} or {mol_id: "smi1;smi2;..."}
    - ground_truth: {mol_id: true_smiles} or {mol_id: true_ik14}
    - Metric: mean(1 / rank of first correct guess; 0 if none in top 25).
    - Guesses are de-duplicated by InChIKey14.
    """
    reciprocal_ranks = []
    for mol_id, true_val in ground_truth.items():
        if not true_val:
            continue
        # Obtain true InChIKey14
        true_ik = true_val if len(str(true_val)) == 14 and "-" not in str(true_val) else ik14(str(true_val))
        if not true_ik:
            continue

        raw_preds = predictions.get(mol_id, [])
        if isinstance(raw_preds, str):
            raw_preds = [s.strip() for s in raw_preds.split(";") if s.strip()]

        seen_keys = set()
        matched_rank = 0
        current_rank = 0

        for cand_smi in raw_preds:
            cand_ik = ik14(cand_smi)
            if not cand_ik or cand_ik in seen_keys:
                continue
            seen_keys.add(cand_ik)
            current_rank += 1
            if cand_ik == true_ik:
                matched_rank = current_rank
                break
            if current_rank >= 25:
                break

        score = 1.0 / matched_rank if matched_rank > 0 else 0.0
        reciprocal_ranks.append(score)

    return float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0

# Adduct shift lookup table (Rule 6: neutral_mass = charge * precursor_mz - shift)
PROTON_MASS = 1.00727646677
ADDUCT_TABLE = {
    "[M+H]+":        (1,  PROTON_MASS),
    "[M+Na]+":       (1,  22.989218),
    "[M+K]+":        (1,  38.963158),
    "[M+NH4]+":      (1,  18.033823),
    "[M]+":          (1, -0.0005486),
    "[M]-":          (1,  0.0005486),
    "[M-H]-":        (1, -PROTON_MASS),
    "[M+FA-H]-":     (1,  44.998201),
    "[M+CH2O2-H]-":  (1,  44.998201),
    "[M+Cl]-":       (1,  34.969402),
    "[M+Br]-":       (1,  78.918885),
    "[M+HAc-H]-":    (1,  59.013851),
    "[M+C2H4O2-H]-": (1,  59.013851),
    "[M+H-H2O]+":    (1, -17.002740),
    "[M-H2O+H]+":    (1, -17.002740),
    "[M-H2O-H]-":    (1, -19.018392),
    "[M+2H]2+":      (2,  2 * PROTON_MASS),
    "[M-2H]2-":      (2, -2 * PROTON_MASS),
    "[M+2Na-H]+":    (1,  44.971160),
}
_unknown_adducts = Counter()

def neutral_mass_from_precursor(pmz: float, adduct: str, polarity: str = None) -> float:
    """Compute neutral monoisotopic mass from precursor m/z and adduct."""
    adduct_str = str(adduct).strip() if pd.notna(adduct) else ""
    if adduct_str in ADDUCT_TABLE:
        charge, shift = ADDUCT_TABLE[adduct_str]
        return float(pmz * charge - shift)
    _unknown_adducts[adduct_str] += 1
    pol = str(polarity).lower() if pd.notna(polarity) else "positive"
    return float(pmz + PROTON_MASS if "neg" in pol else pmz - PROTON_MASS)

def truncate_display(val, max_len=8):
    if val is None:
        return "None"
    if isinstance(val, (list, np.ndarray)):
        arr = list(val)
        return str(arr[:max_len]) + (f" ... ({len(arr)} items)" if len(arr) > max_len else "")
    return str(val)

print(f"Utilities initialized. [{elapsed():.1f}s]")

In [ ]:
# ==============================================================================
# SCHEMA ADAPTER (Automatic Detection & Format Normalization)
# ==============================================================================

class SchemaAdapter:
    """Auto-detects semantic columns across various competition schemas."""
    CANDIDATES = {
        "molecule_id":      ["molecule_id", "mol_id", "compound_id", "inchikey14", "inchikey", "id"],
        "spectrum_id":      ["spectrum_id", "spec_id", "scan_id", "id"],
        "smiles":           ["normalized_smiles", "smiles", "SMILES", "canonical_smiles", "structure", "smi"],
        "precursor_mz":     ["precursor_mz", "precursor_mass", "prec_mz", "mz", "parent_mz", "precursormz"],
        "adduct":           ["adduct", "adduct_type", "precursor_type", "ion_type"],
        "adduct_orig":      ["adduct_orig", "original_adduct"],
        "polarity":         ["ionization_mode", "polarity", "ion_mode", "mode", "ionmode"],
        "collision_energy": ["collision_energy_ev", "collision_energy", "ce", "CE", "energy", "collision_energy_orig"],
        "instrument":       ["instrument_type", "instrument", "inst_type", "instrument_name"],
        "formula":          ["molecular_formula", "formula", "mol_formula", "molecularformula"],
        "inchikey":         ["inchikey", "InChIKey", "inchi_key"],
        "inchikey14":       ["inchikey14", "InChIKey14", "ik14"],
        "mz_array":         ["ms2_mzs", "mz_array", "peaks_mz", "mzs", "fragment_mz", "mz_values"],
        "intensity_array":  ["ms2_normalized_intensities", "intensity_array", "peaks_intensity", "intensities",
                             "normalized_intensities", "intensity_values"],
        "base_peak_intensity": ["base_peak_intensity", "base_peak_int"],
        "num_peaks":        ["num_peaks", "peak_count", "n_peaks"],
        "ingest_lib":       ["ingest_lib", "library", "source_lib", "source"],
    }

    def __init__(self, df: pd.DataFrame, name: str = "dataset"):
        self.name = name
        self.mapping = {}
        self.col_types = {}
        cols_lower = {c.lower(): c for c in df.columns}

        for field_name, candidates in self.CANDIDATES.items():
            found = None
            for cand in candidates:
                if cand in df.columns:
                    found = cand
                    break
                if cand.lower() in cols_lower:
                    found = cols_lower[cand.lower()]
                    break
            if found is not None:
                self.mapping[field_name] = found
                self.col_types[field_name] = self._detect_col_type(df[found])

        # Validate essential columns
        critical = ["precursor_mz", "mz_array", "intensity_array"]
        missing = [c for c in critical if c not in self.mapping]
        if missing:
            raise ValueError(f"SchemaAdapter({name}): Critical columns missing: {missing}. Available: {list(df.columns)}")

        print(f"SchemaAdapter({name}): Successfully mapped {len(self.mapping)} fields:")
        for k, v in self.mapping.items():
            print(f"  {k:22s} -> {v:30s} [{self.col_types[k]}]")

    def _detect_col_type(self, series: pd.Series) -> str:
        sample = series.dropna().iloc[:3] if len(series.dropna()) > 0 else pd.Series()
        if len(sample) == 0:
            return "empty"
        val = sample.iloc[0]
        if isinstance(val, (list, np.ndarray)):
            if len(val) > 0 and isinstance(val[0], dict):
                return "list_of_structs"
            return "array"
        if isinstance(val, str):
            try:
                parsed = json.loads(val)
                if isinstance(parsed, list):
                    return "json_array"
                return "json_dict"
            except (json.JSONDecodeError, TypeError):
                return "string"
        if isinstance(val, (int, float, np.number)):
            return "numeric"
        return f"object({type(val).__name__})"

    def col(self, field_name: str):
        return self.mapping.get(field_name, None)

    def has(self, field_name: str) -> bool:
        return field_name in self.mapping

    def to_dict(self):
        return {"mapping": self.mapping, "col_types": self.col_types}

## 1. Schema Discovery & Data Ingestion
Load train and test parquet files from DATA_DIR (or fallback locations), detect schemas using `SchemaAdapter`, and inspect sample rows.

In [ ]:
# ==============================================================================
# DATA LOADING & RAW INSPECTION
# ==============================================================================
def resolve_data_paths(cfg: CFG):
    """Resolve train and test data paths with multi-level fallbacks."""
    # 1. Configured DATA_DIR
    p_tr = os.path.join(cfg.data_dir, "train.parquet")
    p_te = os.path.join(cfg.data_dir, "test.parquet")
    if os.path.exists(p_tr) and os.path.exists(p_te):
        return p_tr, p_te

    # 2. Kaggle competition input glob
    k_tr = sorted(glob("/kaggle/input/**/train.parquet", recursive=True))
    k_te = sorted(glob("/kaggle/input/**/test.parquet", recursive=True))
    if k_tr and k_te:
        return k_tr[0], k_te[0]

    # 3. Local workspace train/test.parquet
    l_tr = sorted(glob("**/train.parquet", recursive=True))
    l_te = sorted(glob("**/test.parquet", recursive=True))
    if l_tr and l_te:
        return l_tr[0], l_te[0]

    # 4. Local workspace head100 files fallback
    h_tr = sorted(glob("**/train_head100.parquet", recursive=True))
    h_te = sorted(glob("**/test_head100.parquet", recursive=True))
    if h_tr and h_te:
        return h_tr[0], h_te[0]

    raise FileNotFoundError(f"Could not locate train and test datasets in {cfg.data_dir} or workspace fallbacks.")

train_path, test_path = resolve_data_paths(cfg)
print(f"Loading Train from: {train_path}")
print(f"Loading Test  from: {test_path}")

train_raw = pd.read_parquet(train_path)
test_raw  = pd.read_parquet(test_path)

if cfg.SMOKE_TEST:
    train_raw = train_raw.head(cfg.smoke_n_rows).copy()
    test_raw  = test_raw.head(min(cfg.smoke_n_rows, len(test_raw))).copy()
    print(f"SMOKE_TEST Active: using {len(train_raw)} train and {len(test_raw)} test rows.")

print(f"Train Shape: {train_raw.shape}, Memory: {train_raw.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"Test  Shape: {test_raw.shape},  Memory: {test_raw.memory_usage(deep=True).sum() / 1e6:.2f} MB")

# Schema adapters
adapter_train = SchemaAdapter(train_raw, name="train")
adapter_test  = SchemaAdapter(test_raw, name="test")

# Display 3 sample rows with truncated arrays
print("\n--- Train Sample (3 Rows) ---")
tr_sample = train_raw.head(3).copy()
for col in tr_sample.columns:
    tr_sample[col] = tr_sample[col].apply(lambda x: truncate_display(x, cfg.truncate_array_len))
print(tr_sample.to_string())

print("\n--- Test Sample (3 Rows) ---")
te_sample = test_raw.head(3).copy()
for col in te_sample.columns:
    te_sample[col] = te_sample[col].apply(lambda x: truncate_display(x, cfg.truncate_array_len))
print(te_sample.to_string())

# Ensure output directory exists
def resolve_output_dir(path: str) -> str:
    try:
        os.makedirs(path, exist_ok=True)
        return path
    except (PermissionError, OSError):
        local_dir = os.path.join(".", "artifacts", "eda")
        os.makedirs(local_dir, exist_ok=True)
        return local_dir

OUTPUT_DIR = resolve_output_dir(cfg.output_dir)
print(f"\nArtifact output directory: {OUTPUT_DIR}")

In [ ]:
# ==============================================================================
# SAVE DETECTED SCHEMA ARTIFACT
# ==============================================================================
schema_artifact = {
    "train": adapter_train.to_dict(),
    "test":  adapter_test.to_dict(),
    "train_columns": list(train_raw.columns),
    "test_columns":  list(test_raw.columns),
    "train_shape": list(train_raw.shape),
    "test_shape":  list(test_raw.shape),
    "train_only_columns": sorted(list(set(train_raw.columns) - set(test_raw.columns))),
    "test_only_columns":  sorted(list(set(test_raw.columns) - set(train_raw.columns))),
    "timestamp": datetime.now().isoformat(),
}

schema_path = os.path.join(OUTPUT_DIR, "schema.json")
with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(schema_artifact, f, indent=2)

print(f"Saved: {schema_path}")
print(f"Train-only columns (Labels & Metadata): {schema_artifact['train_only_columns']}")
print(f"Test-only columns: {schema_artifact['test_only_columns']}")

## 2. Vectorized Spectral Statistics & Train vs Test Distribution Comparison
Compute spectrum-level statistics using vectorized list operations (peaks per spectrum, precursor m/z range, adduct frequencies, collision energies, and precursor peak presence) and perform Kolmogorov-Smirnov distribution tests between train and test.

In [ ]:
# ==============================================================================
# VECTORIZED SPECTRAL STATISTICS & DISTRIBUTION COMPARISON
# ==============================================================================
check_budget()

def compute_spectral_stats_vectorized(df: pd.DataFrame, adapter: SchemaAdapter, name: str = "train") -> dict:
    stats = {"name": name, "n_rows": int(len(df))}

    # Unique molecules & spectra per molecule
    mol_col = adapter.col("molecule_id")
    if mol_col and mol_col in df.columns:
        stats["n_unique_molecules"] = int(df[mol_col].nunique())
        spm = df.groupby(mol_col).size()
        stats["spectra_per_molecule"] = {
            "mean": float(spm.mean()),
            "median": float(spm.median()),
            "min": int(spm.min()),
            "max": int(spm.max()),
        }
    else:
        stats["n_unique_molecules"] = int(len(df))
        stats["spectra_per_molecule"] = {"mean": 1.0, "median": 1.0, "min": 1, "max": 1}

    # Vectorized peak count (using num_peaks column if present, else fast list comprehension)
    if adapter.has("num_peaks") and adapter.col("num_peaks") in df.columns:
        peaks_counts = df[adapter.col("num_peaks")].to_numpy(dtype=np.int64)
    else:
        mz_col = adapter.col("mz_array")
        raw_mzs = df[mz_col].tolist()
        peaks_counts = np.array([len(x) if hasattr(x, '__len__') else 0 for x in raw_mzs], dtype=np.int64)

    stats["peaks_per_spectrum"] = {
        "mean": float(np.mean(peaks_counts)),
        "median": float(np.median(peaks_counts)),
        "min": int(np.min(peaks_counts)),
        "max": int(np.max(peaks_counts)),
        "q95": float(np.percentile(peaks_counts, 95)),
        "q99": float(np.percentile(peaks_counts, 99)),
    }
    stats["_peaks_counts"] = peaks_counts

    # Precursor m/z
    pmz_col = adapter.col("precursor_mz")
    pmz_vals = df[pmz_col].dropna().to_numpy(dtype=np.float64)
    stats["precursor_mz"] = {
        "min": float(np.min(pmz_vals)),
        "max": float(np.max(pmz_vals)),
        "mean": float(np.mean(pmz_vals)),
        "median": float(np.median(pmz_vals)),
    }
    stats["_precursor_mz"] = pmz_vals

    # Adduct and polarity counts
    adduct_col = adapter.col("adduct")
    if adduct_col and adduct_col in df.columns:
        stats["adduct_counts"] = df[adduct_col].value_counts().to_dict()
    pol_col = adapter.col("polarity")
    if pol_col and pol_col in df.columns:
        stats["polarity_counts"] = df[pol_col].value_counts().to_dict()

    # Collision energy
    ce_col = adapter.col("collision_energy")
    if ce_col and ce_col in df.columns:
        ce_extracted = []
        for v in df[ce_col].dropna().tolist():
            if isinstance(v, (list, np.ndarray)) and len(v) > 0:
                ce_extracted.append(float(v[0]))
            elif isinstance(v, (int, float, np.number)):
                ce_extracted.append(float(v))
        if ce_extracted:
            stats["ce_first_value"] = {
                "mean": float(np.mean(ce_extracted)),
                "median": float(np.median(ce_extracted)),
            }

    inst_col = adapter.col("instrument")
    if inst_col and inst_col in df.columns:
        stats["instrument_counts"] = df[inst_col].fillna("None/Unknown").value_counts().to_dict()

    # Precursor peak presence check (±0.5 Da) using zipped native lists
    sample_size = min(5000, len(df))
    sub = df.sample(sample_size, random_state=cfg.seed)
    mz_list = sub[adapter.col("mz_array")].tolist()
    pmz_list = sub[pmz_col].tolist()
    int_list = sub[adapter.col("intensity_array")].tolist()

    has_prec = 0
    all_sampled_intensities = []
    for mzs, pmz, ints in zip(mz_list, pmz_list, int_list):
        if hasattr(mzs, '__len__') and len(mzs) > 0 and pd.notna(pmz):
            arr = np.array(mzs, dtype=np.float64)
            if np.any(np.abs(arr - pmz) < 0.5):
                has_prec += 1
        if hasattr(ints, '__len__') and len(ints) > 0:
            all_sampled_intensities.extend(list(ints)[:100])

    stats["frac_precursor_peak_present"] = float(has_prec / sample_size) if sample_size > 0 else 0.0

    # Accelerate intensity quantiles via CUDA if available
    all_sampled_intensities = np.array(all_sampled_intensities, dtype=np.float64)
    if len(all_sampled_intensities) > 0:
        if CUDA_AVAILABLE:
            t_int = torch.tensor(all_sampled_intensities, device='cuda:0', dtype=torch.float32)
            stats["intensity_distribution"] = {
                "mean": float(t_int.mean().item()),
                "median": float(t_int.median().item()),
                "frac_above_0.1": float((t_int > 0.1).float().mean().item()),
                "frac_above_0.5": float((t_int > 0.5).float().mean().item()),
            }
        else:
            stats["intensity_distribution"] = {
                "mean": float(np.mean(all_sampled_intensities)),
                "median": float(np.median(all_sampled_intensities)),
                "frac_above_0.1": float(np.mean(all_sampled_intensities > 0.1)),
                "frac_above_0.5": float(np.mean(all_sampled_intensities > 0.5)),
            }

    return stats

train_spectral_stats = compute_spectral_stats_vectorized(train_raw, adapter_train, "train")
test_spectral_stats  = compute_spectral_stats_vectorized(test_raw, adapter_test, "test")

# Kolmogorov-Smirnov distribution comparison
ks_pmz = sp_stats.ks_2samp(train_spectral_stats["_precursor_mz"], test_spectral_stats["_precursor_mz"])
ks_peaks = sp_stats.ks_2samp(train_spectral_stats["_peaks_counts"], test_spectral_stats["_peaks_counts"])

distribution_comparison = {
    "ks_precursor_mz": {"statistic": float(ks_pmz.statistic), "pvalue": float(ks_pmz.pvalue)},
    "ks_peaks_per_spec": {"statistic": float(ks_peaks.statistic), "pvalue": float(ks_peaks.pvalue)},
}

print(f"Train: {train_spectral_stats['n_rows']} rows, {train_spectral_stats['n_unique_molecules']} molecules, "
      f"Mean peaks/spec: {train_spectral_stats['peaks_per_spectrum']['mean']:.1f}")
print(f"Test:  {test_spectral_stats['n_rows']} rows, {test_spectral_stats['n_unique_molecules']} molecules, "
      f"Mean peaks/spec: {test_spectral_stats['peaks_per_spectrum']['mean']:.1f}")
print(f"KS-Test Precursor m/z: stat={ks_pmz.statistic:.4f}, p={ks_pmz.pvalue:.4e}")
print(f"KS-Test #Peaks/Spec:   stat={ks_peaks.statistic:.4f}, p={ks_peaks.pvalue:.4e}")
print(f"Precursor peak present in fragment list: Train={train_spectral_stats['frac_precursor_peak_present']:.1%}, "
      f"Test={test_spectral_stats['frac_precursor_peak_present']:.1%}")

## 3. High-Throughput Molecule Profiling (Multiprocessing across 4 vCPUs)
Profile unique structures using `joblib` multiprocessing:
- Native Python lists (`smiles_list = train_mols[smi_col].tolist()`) eliminate pandas `iterrows()` overhead.
- Worker-local RDKit enumerators circumvent Boost Python pickling limitations.
- Measure RDKit parse rate, heavy atoms, exact MW, element sets, Murcko scaffolds, and adduct neutrality error within 10 ppm.

In [ ]:
# ==============================================================================
# HIGH-THROUGHPUT MOLECULE PROFILING (JOBLIB MULTIPROCESSING ACROSS 4 vCPUs)
# ==============================================================================
check_budget()

smi_col = adapter_train.col("smiles")
mol_id_col = adapter_train.col("molecule_id")
pmz_col = adapter_train.col("precursor_mz")
adduct_col = adapter_train.col("adduct")
pol_col = adapter_train.col("polarity")
ik_col = adapter_train.col("inchikey14")

# Deduplicate to unique molecules
train_mols = train_raw.drop_duplicates(subset=[smi_col]).copy()
print(f"Profiling {len(train_mols)} unique molecules using {cfg.n_cpu_workers} CPU worker processes...")

# Extract native Python lists (avoids all pandas indexing overhead)
smiles_list = train_mols[smi_col].tolist()
pmz_list = train_mols[pmz_col].tolist()
adduct_list = train_mols[adduct_col].tolist() if adduct_col and adduct_col in train_mols.columns else ["[M+H]+"] * len(train_mols)
pol_list = train_mols[pol_col].tolist() if pol_col and pol_col in train_mols.columns else ["positive"] * len(train_mols)
ik14_precomputed = train_mols[ik_col].tolist() if ik_col and ik_col in train_mols.columns else [None] * len(train_mols)

# Pack into plain tuples
tuple_items = list(zip(smiles_list, pmz_list, adduct_list, pol_list, ik14_precomputed))

# Multiprocessing single molecule worker function
def _profile_single_molecule(item):
    smi, pmz, adduct, pol, pre_ik = item
    if not smi or not isinstance(smi, str):
        return (False, None, None, None, None, False, False, None, None, None)
    mol = MolFromSmiles(smi)
    if mol is None:
        return (False, None, None, None, None, False, False, None, None, None)

    h_atoms = mol.GetNumHeavyAtoms()
    mw = Descriptors.ExactMolWt(mol)
    elems = [atom.GetSymbol() for atom in mol.GetAtoms()]
    formula = rdMolDescriptors.CalcMolFormula(mol)
    is_salt = "." in smi
    is_charged = (Chem.GetFormalCharge(mol) != 0)

    # InChIKey14: reuse precomputed if available, else compute locally
    if pre_ik and len(str(pre_ik)) == 14 and "-" not in str(pre_ik):
        key14 = str(pre_ik)
    else:
        key14 = ik14(smi)

    # Scaffold
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        scaffold_smi = MolToSmiles(scaffold)
    except Exception:
        scaffold_smi = "_FAIL_"

    # Adduct neutral mass error
    calc_neutral = neutral_mass_from_precursor(pmz, adduct, pol)
    ppm_err = abs(calc_neutral - mw) / mw * 1e6 if mw > 0 else 1e6

    return (True, h_atoms, mw, elems, formula, is_salt, is_charged, key14, scaffold_smi, ppm_err)

t_start = time.time()
try:
    flat_results = joblib.Parallel(n_jobs=cfg.n_cpu_workers, prefer="threads", batch_size=200)(
        joblib.delayed(_profile_single_molecule)(item) for item in tuple_items
    )
except Exception:
    flat_results = [_profile_single_molecule(item) for item in tuple_items]
print(f"Multiprocessing complete in {time.time() - t_start:.2f}s ({len(flat_results)} molecules profiled).")

# Aggregate results
parse_ok = 0
parse_fail = 0
heavy_atoms = []
exact_mws = []
element_counter = Counter()
formula_counter = Counter()
scaffold_counter = Counter()
unique_ik14_set = set()
multi_frag_count = 0
charged_count = 0
ppm_errors = []

for r in flat_results:
    if not r[0]:
        parse_fail += 1
        continue
    parse_ok += 1
    _, h_atoms, mw, elems, formula, is_salt, is_charged, key14, scaffold_smi, ppm_err = r
    heavy_atoms.append(h_atoms)
    exact_mws.append(mw)
    for el in elems:
        element_counter[el] += 1
    formula_counter[formula] += 1
    scaffold_counter[scaffold_smi] += 1
    if is_salt:
        multi_frag_count += 1
    if is_charged:
        charged_count += 1
    if key14:
        unique_ik14_set.add(key14)
    ppm_errors.append(ppm_err)

ppm_errors = np.array(ppm_errors, dtype=np.float64)
n_mols = parse_ok + parse_fail
mass_matched_10ppm = int(np.sum(ppm_errors <= cfg.ppm_tolerance))

molecule_stats = {
    "n_molecules": n_mols,
    "rdkit_parse_rate": float(parse_ok / n_mols) if n_mols > 0 else 0.0,
    "heavy_atom_count": {
        "mean": float(np.mean(heavy_atoms)),
        "median": float(np.median(heavy_atoms)),
        "min": int(np.min(heavy_atoms)),
        "max": int(np.max(heavy_atoms)),
    },
    "molecular_weight": {
        "mean": float(np.mean(exact_mws)),
        "median": float(np.median(exact_mws)),
        "min": float(np.min(exact_mws)),
        "max": float(np.max(exact_mws)),
    },
    "element_set": dict(element_counter.most_common(20)),
    "top_formulas": dict(formula_counter.most_common(20)),
    "top_scaffolds": dict(scaffold_counter.most_common(20)),
    "n_unique_ik14": len(unique_ik14_set),
    "n_unique_scaffolds": len(scaffold_counter),
    "frac_multi_fragment_salts": float(multi_frag_count / n_mols) if n_mols > 0 else 0.0,
    "frac_charged": float(charged_count / n_mols) if n_mols > 0 else 0.0,
    "mass_sanity_check": {
        "n_checked": len(ppm_errors),
        "n_matched_within_10ppm": mass_matched_10ppm,
        "frac_matched": float(mass_matched_10ppm / len(ppm_errors)) if len(ppm_errors) > 0 else 0.0,
        "median_ppm_error": float(np.median(ppm_errors)),
        "q95_ppm_error": float(np.percentile(ppm_errors, 95)),
    }
}

print(f"RDKit parse rate: {molecule_stats['rdkit_parse_rate']:.2%} ({parse_ok}/{n_mols})")
print(f"Molecular Weight: mean={molecule_stats['molecular_weight']['mean']:.2f}, "
      f"median={molecule_stats['molecular_weight']['median']:.2f} Da")
print(f"Heavy Atoms:      mean={molecule_stats['heavy_atom_count']['mean']:.1f}, "
      f"max={molecule_stats['heavy_atom_count']['max']}")
print(f"Unique InChIKey14: {molecule_stats['n_unique_ik14']} | Unique Scaffolds: {molecule_stats['n_unique_scaffolds']}")
print(f"Multi-fragment (salts): {molecule_stats['frac_multi_fragment_salts']:.2%} | Charged: {molecule_stats['frac_charged']:.2%}")
print(f"Adduct Sanity Check: {molecule_stats['mass_sanity_check']['frac_matched']:.2%} match within {cfg.ppm_tolerance} ppm "
      f"(Median error: {molecule_stats['mass_sanity_check']['median_ppm_error']:.3f} ppm)")

## 4. Dual T4 GPU Accelerated Overlap Analysis
Compute neutral mass overlap between test and train libraries:
- Vectorized binary search (`torch.searchsorted`) sharded across `cuda:0` and `cuda:1` when available (falling back to vectorized CPU).
- Determines the exact theoretical MRR@25 ceiling for library mass lookup.

In [ ]:
# ==============================================================================
# DUAL T4 GPU ACCELERATED OVERLAP ANALYSIS (TEST vs TRAIN LIBRARY)
# ==============================================================================
check_budget()

# Precursor & adduct columns
test_mol_col = adapter_test.col("molecule_id")
test_pmz_col = adapter_test.col("precursor_mz")
test_adduct_col = adapter_test.col("adduct")
test_pol_col = adapter_test.col("polarity")

# Vectorized neutral mass for train (using unique molecules)
tr_pmz_arr = train_mols[pmz_col].to_numpy(dtype=np.float64)
tr_add_list = train_mols[adduct_col].tolist() if adduct_col and adduct_col in train_mols.columns else ["[M+H]+"] * len(train_mols)
tr_pol_list = train_mols[pol_col].tolist() if pol_col and pol_col in train_mols.columns else ["positive"] * len(train_mols)

tr_shifts = [ADDUCT_TABLE.get(str(a).strip(), (1, PROTON_MASS if "pos" in str(p).lower() else -PROTON_MASS))
             for a, p in zip(tr_add_list, tr_pol_list)]
tr_charges = np.array([c for c, s in tr_shifts], dtype=np.float64)
tr_deltas  = np.array([s for c, s in tr_shifts], dtype=np.float64)
train_neutral_sorted = np.sort(tr_pmz_arr * tr_charges - tr_deltas)

# Vectorized neutral mass for test
test_mols = test_raw.drop_duplicates(subset=[test_mol_col]).copy()
te_pmz_arr = test_mols[test_pmz_col].to_numpy(dtype=np.float64)
te_add_list = test_mols[test_adduct_col].tolist() if test_adduct_col and test_adduct_col in test_mols.columns else ["[M+H]+"] * len(test_mols)
te_pol_list = test_mols[test_pol_col].tolist() if test_pol_col and test_pol_col in test_mols.columns else ["positive"] * len(test_mols)

te_shifts = [ADDUCT_TABLE.get(str(a).strip(), (1, PROTON_MASS if "pos" in str(p).lower() else -PROTON_MASS))
             for a, p in zip(te_add_list, te_pol_list)]
te_charges = np.array([c for c, s in te_shifts], dtype=np.float64)
te_deltas  = np.array([s for c, s in te_shifts], dtype=np.float64)
test_neutral_masses = te_pmz_arr * te_charges - te_deltas

# Accelerated Overlap Search across Dual T4 GPUs / CPU
def accelerated_mass_overlap(train_sorted: np.ndarray, test_query: np.ndarray, ppm_tol: float = 5.0) -> int:
    if CUDA_AVAILABLE and GPU_COUNT >= 2:
        print(f"Distributing mass overlap search across Dual T4 GPUs (cuda:0 & cuda:1)...")
        mid = len(test_query) // 2
        # Shard 0 on cuda:0
        t0 = torch.tensor(train_sorted, device='cuda:0', dtype=torch.float64)
        q0 = torch.tensor(test_query[:mid], device='cuda:0', dtype=torch.float64)
        tol0 = q0 * (ppm_tol / 1e6)
        lo0 = torch.searchsorted(t0, q0 - tol0)
        hi0 = torch.searchsorted(t0, q0 + tol0)
        m0 = int((hi0 > lo0).sum().item())

        # Shard 1 on cuda:1
        t1 = torch.tensor(train_sorted, device='cuda:1', dtype=torch.float64)
        q1 = torch.tensor(test_query[mid:], device='cuda:1', dtype=torch.float64)
        tol1 = q1 * (ppm_tol / 1e6)
        lo1 = torch.searchsorted(t1, q1 - tol1)
        hi1 = torch.searchsorted(t1, q1 + tol1)
        m1 = int((hi1 > lo1).sum().item())
        return m0 + m1
    elif CUDA_AVAILABLE and GPU_COUNT == 1:
        print("Executing mass overlap search on cuda:0...")
        t = torch.tensor(train_sorted, device='cuda:0', dtype=torch.float64)
        q = torch.tensor(test_query, device='cuda:0', dtype=torch.float64)
        tol = q * (ppm_tol / 1e6)
        lo = torch.searchsorted(t, q - tol)
        hi = torch.searchsorted(t, q + tol)
        return int((hi > lo).sum().item())
    else:
        # Vectorized PyTorch CPU search
        t = torch.tensor(train_sorted, dtype=torch.float64)
        q = torch.tensor(test_query, dtype=torch.float64)
        tol = q * (ppm_tol / 1e6)
        lo = torch.searchsorted(t, q - tol)
        hi = torch.searchsorted(t, q + tol)
        return int((hi > lo).sum().item())

n_test_covered = accelerated_mass_overlap(train_neutral_sorted, test_neutral_masses, cfg.mass_match_ppm)
overlap_fraction = float(n_test_covered / len(test_neutral_masses)) if len(test_neutral_masses) > 0 else 0.0

overlap_analysis = {
    "n_train_molecules": len(train_neutral_sorted),
    "n_test_molecules": len(test_neutral_masses),
    "ppm_tolerance": cfg.mass_match_ppm,
    "n_test_with_mass_match": int(n_test_covered),
    "frac_test_mass_covered": overlap_fraction,
    "theoretical_max_library_lookup_mrr": overlap_fraction,
}

print(f"Overlap Analysis (within +/-{cfg.mass_match_ppm} ppm):")
print(f"  Test molecules evaluated: {len(test_neutral_masses)}")
print(f"  Test molecules matching ANY train neutral mass: {n_test_covered} ({overlap_fraction:.2%})")
print(f"  Theoretical ceiling for train-library mass retrieval: {overlap_fraction:.2%}")

## 5. Diagnostic Visualizations & Representative Spectra
Generate 5 distribution charts (precursor m/z, peaks per spectrum, adducts, molecular properties, and ppm error) and 1 panel of 3 representative MS/MS spectra annotated with precursor lines.

In [ ]:
# ==============================================================================
# DIAGNOSTIC VISUALIZATIONS (5 PLOTS)
# ==============================================================================
check_budget()
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
CLR_TR = "#1f77b4"
CLR_TE = "#ff7f0e"

# 1. Precursor m/z Distribution
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(train_spectral_stats["_precursor_mz"], bins=150, alpha=0.55, color=CLR_TR, label="Train", density=True)
ax.hist(test_spectral_stats["_precursor_mz"], bins=150, alpha=0.55, color=CLR_TE, label="Test", density=True)
ax.set_title("Precursor m/z Distribution (Train vs Test)")
ax.set_xlabel("Precursor m/z (Da)")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "01_precursor_mz.png"), dpi=150)
plt.close(fig)

# 2. Peaks per Spectrum Distribution
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(train_spectral_stats["_peaks_counts"], bins=100, alpha=0.55, color=CLR_TR, label="Train", density=True)
ax.hist(test_spectral_stats["_peaks_counts"], bins=100, alpha=0.55, color=CLR_TE, label="Test", density=True)
ax.set_title("Peaks per Spectrum Distribution (Train vs Test)")
ax.set_xlabel("Number of Peaks")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "02_peaks_per_spectrum.png"), dpi=150)
plt.close(fig)

# 3. Adduct & Polarity Distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
if "adduct_counts" in train_spectral_stats:
    top_adducts = dict(sorted(train_spectral_stats["adduct_counts"].items(), key=lambda x: -x[1])[:8])
    axes[0].barh(list(top_adducts.keys()), list(top_adducts.values()), color=CLR_TR, alpha=0.8)
    axes[0].set_title("Top Train Adducts")
    axes[0].invert_yaxis()
if "polarity_counts" in train_spectral_stats:
    pol = train_spectral_stats["polarity_counts"]
    axes[1].bar(list(pol.keys()), list(pol.values()), color=[CLR_TR, CLR_TE][:len(pol)], alpha=0.8)
    axes[1].set_title("Polarity Split (Train)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "03_adduct_polarity.png"), dpi=150)
plt.close(fig)

# 4. Molecular Weight & Heavy Atom Distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(exact_mws, bins=80, color=CLR_TR, alpha=0.75)
axes[0].set_title("Train Exact Molecular Weight")
axes[0].set_xlabel("MW (Da)")
axes[1].hist(heavy_atoms, bins=60, color="#2ca02c", alpha=0.75)
axes[1].set_title("Train Heavy Atom Count")
axes[1].set_xlabel("# Heavy Atoms")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "04_mol_properties.png"), dpi=150)
plt.close(fig)

# 5. Adduct Sanity PPM Error
fig, ax = plt.subplots(figsize=(9, 4.5))
ppm_sub = ppm_errors[ppm_errors < 50.0]
ax.hist(ppm_sub, bins=100, color="#2ca02c", alpha=0.75)
ax.axvline(10.0, color="red", linestyle="--", label="10 ppm cut-off")
ax.set_title(f"Adduct Sanity Check: PPM Error ({molecule_stats['mass_sanity_check']['frac_matched']:.1%} <= 10 ppm)")
ax.set_xlabel("PPM Error")
ax.set_ylabel("Count")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "05_ppm_error.png"), dpi=150)
plt.close(fig)

print(f"5 Statistical Figures saved to {FIG_DIR}")

In [ ]:
# ==============================================================================
# EXAMPLE SPECTRA PLOTS (3 REPRESENTATIVE SPECTRA)
# ==============================================================================
check_budget()

mz_col = adapter_train.col("mz_array")
int_col = adapter_train.col("intensity_array")
pmz_col = adapter_train.col("precursor_mz")
smi_col = adapter_train.col("smiles")
adduct_col = adapter_train.col("adduct")

# Select 3 distinct spectra with good peak counts using fast native iteration
chosen_spectra = []
for idx, (mzs, ints, pmz, add, smi) in enumerate(zip(
    train_raw[mz_col].tolist(),
    train_raw[int_col].tolist(),
    train_raw[pmz_col].tolist(),
    train_raw[adduct_col].tolist() if adduct_col and adduct_col in train_raw.columns else ["?"] * len(train_raw),
    train_raw[smi_col].tolist() if smi_col and smi_col in train_raw.columns else [""] * len(train_raw),
)):
    if hasattr(mzs, '__len__') and 15 <= len(mzs) <= 250:
        chosen_spectra.append((mzs, ints, pmz, add, smi))
        if len(chosen_spectra) == 3:
            break

fig, axes = plt.subplots(3, 1, figsize=(11, 8.5), sharex=False)
for idx, (mzs, ints, pmz, adduct, smi) in enumerate(chosen_spectra):
    ax = axes[idx]
    mzs_arr = np.array(mzs, dtype=np.float64)
    ints_arr = np.array(ints, dtype=np.float64)
    smi_disp = str(smi)[:45] + "..." if smi else ""

    # Stem-like peaks
    markerline, stemlines, baseline = ax.stem(mzs_arr, ints_arr, linefmt="-", markerfmt=" ", basefmt="-")
    stemlines.set_color(CLR_TR)
    stemlines.set_linewidth(0.9)
    baseline.set_color("black")
    baseline.set_linewidth(0.5)

    # Precursor indicator
    ax.axvline(float(pmz), color="red", linestyle="--", linewidth=1.2, label=f"Precursor m/z: {float(pmz):.4f}")
    ax.set_title(f"Spectrum #{idx+1}: {smi_disp} | Adduct: {adduct} | {len(mzs)} peaks", fontsize=10)
    ax.set_ylabel("Norm. Intensity")
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Fragment m/z (Da)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "06_example_spectra.png"), dpi=150)
plt.close(fig)

print(f"Example spectra figure saved to {os.path.join(FIG_DIR, '06_example_spectra.png')}")

## 6. Validation Section: MRR@25 Evaluation with Dual GPU Candidate Ranking
Demonstrate competition metric `compute_mrr25`:
- Assertion tests confirming keto-enol tautomer equivalence and InChIKey14 deduplication.
- Dual T4 GPU accelerated mass-proximity top-25 retrieval on a held-out split of molecules.

In [ ]:
# ==============================================================================
# VALIDATION SECTION: MRR@25 SCORER & DUAL GPU CANDIDATE RETRIEVAL
# ==============================================================================
check_budget()

# Verify metric scorer on synthetic test assertions
test_gt = {
    "mol_1": "c1ccccc1O",       # Phenol
    "mol_2": "CC(=O)O",          # Acetic acid
    "mol_3": "c1ccccc1N",       # Aniline
    "mol_4": "CC(=O)Oc1ccccc1C(=O)O" # Aspirin
}

test_preds = {
    # Rank 1 correct -> 1.0
    "mol_1": ["c1ccccc1O", "CCO"],
    # Rank 2 correct -> 0.5
    "mol_2": ["CCO", "CC(=O)O"],
    # Tautomer canonicalization invariance test (enol vs keto should match!) -> 1.0
    "mol_3": ["C1=CC=C(C=C1)N"],
    # Missed in top 25 -> 0.0
    "mol_4": ["c1ccccc1", "c1ccccc1Cl"]
}

demo_mrr = compute_mrr25(test_preds, test_gt)
expected_mrr = (1.0 + 0.5 + 1.0 + 0.0) / 4.0
assert abs(demo_mrr - expected_mrr) < 1e-6, f"MRR scorer discrepancy: {demo_mrr} != {expected_mrr}"

# Evaluate a baseline mass-retrieval ranking on a held-out split of train molecules
split_n = min(50, len(train_mols))
val_smiles = smiles_list[:split_n]
val_gt = {f"val_{i}": smi for i, smi in enumerate(val_smiles)}

# Dual T4 GPU accelerated top-25 nearest mass retrieval
def compute_top25_candidates_gpu(train_mw_list: list, val_mw_list: list) -> np.ndarray:
    t_mw = np.array(train_mw_list, dtype=np.float32)
    v_mw = np.array(val_mw_list, dtype=np.float32)
    k = min(25, len(t_mw))

    if CUDA_AVAILABLE and GPU_COUNT >= 2:
        print(f"Computing top-{k} mass candidate retrieval sharded across Dual T4 GPUs...")
        mid = len(v_mw) // 2
        # GPU 0
        t0 = torch.tensor(t_mw, device='cuda:0', dtype=torch.float32)
        v0 = torch.tensor(v_mw[:mid], device='cuda:0', dtype=torch.float32)
        diff0 = torch.abs(v0[:, None] - t0[None, :])
        top0 = torch.topk(-diff0, k=k, dim=-1).indices.cpu().numpy()

        # GPU 1
        t1 = torch.tensor(t_mw, device='cuda:1', dtype=torch.float32)
        v1 = torch.tensor(v_mw[mid:], device='cuda:1', dtype=torch.float32)
        diff1 = torch.abs(v1[:, None] - t1[None, :])
        top1 = torch.topk(-diff1, k=k, dim=-1).indices.cpu().numpy()
        return np.vstack([top0, top1]) if len(top0) > 0 and len(top1) > 0 else (top0 if len(top0) > 0 else top1)
    elif CUDA_AVAILABLE and GPU_COUNT == 1:
        print(f"Computing top-{k} mass candidate retrieval on cuda:0...")
        t = torch.tensor(t_mw, device='cuda:0', dtype=torch.float32)
        v = torch.tensor(v_mw, device='cuda:0', dtype=torch.float32)
        diff = torch.abs(v[:, None] - t[None, :])
        return torch.topk(-diff, k=k, dim=-1).indices.cpu().numpy()
    else:
        # PyTorch vectorized CPU tensor
        t = torch.tensor(t_mw, dtype=torch.float32)
        v = torch.tensor(v_mw, dtype=torch.float32)
        diff = torch.abs(v[:, None] - t[None, :])
        return torch.topk(-diff, k=k, dim=-1).indices.numpy()

top25_indices = compute_top25_candidates_gpu(exact_mws, exact_mws[:split_n])
val_preds = {
    f"val_{i}": [smiles_list[idx] for idx in top25_indices[i]]
    for i in range(split_n)
}

val_mrr = compute_mrr25(val_preds, val_gt)
print("MRR@25 Scorer Validation Verified Successfully!")
print(f"  Demo 4-molecule assertion score: {demo_mrr:.4f} (Expected: {expected_mrr:.4f})")
print(f"  Held-out Mass-Proximity Baseline MRR@25 ({split_n} molecules): {val_mrr:.4f}")

## 7. Artifact Saving: Normalized Clean Parquets & Metadata
Export `train_clean.parquet` and `test_clean.parquet` via vectorized array construction, save serializable `stats.json`, and write the stage `manifest.json`.

In [ ]:
# ==============================================================================
# ARTIFACT SAVING: NORMALIZED CLEAN PARQUETS (PURE VECTORIZED COLUMNAR)
# ==============================================================================
check_budget()

charge_map = {k: v[0] for k, v in ADDUCT_TABLE.items()}
shift_map  = {k: v[1] for k, v in ADDUCT_TABLE.items()}

def build_normalized_parquet_fast(df: pd.DataFrame, adapter: SchemaAdapter, is_train: bool = True) -> pd.DataFrame:
    """
    Pure vectorized and zero-copy columnar parquet construction:
    - Direct series assignment for mz (ms2_mzs) and inten (ms2_normalized_intensities).
    - Direct reuse of pre-existing inchikey14 without slow sequential recomputation.
    - Vectorized neutral mass arithmetic (precursor_mz * charge - shift).
    - Train: molecule_id, smiles, ik14, precursor_mz, adduct, polarity, ce, instrument, mz, inten, neutral_mass
    - Test:  molecule_id, precursor_mz, adduct, polarity, ce, instrument, mz, inten, neutral_mass
    """
    n_rows = len(df)
    out = pd.DataFrame(index=df.index)

    # 1. molecule_id
    mol_col = adapter.col("molecule_id")
    if mol_col and mol_col in df.columns:
        out["molecule_id"] = df[mol_col].astype(str)
    elif "inchikey14" in df.columns:
        out["molecule_id"] = df["inchikey14"].astype(str)
    elif is_train and adapter.col("smiles") and adapter.col("smiles") in df.columns:
        out["molecule_id"] = df[adapter.col("smiles")].astype(str)
    else:
        out["molecule_id"] = [f"{'train' if is_train else 'test'}_mol_{i}" for i in range(n_rows)]

    # 2. precursor_mz
    pmz_c = adapter.col("precursor_mz")
    out["precursor_mz"] = pd.to_numeric(df[pmz_c], errors="coerce").fillna(0.0).astype(np.float64)

    # 3. adduct
    add_c = adapter.col("adduct")
    if add_c and add_c in df.columns:
        out["adduct"] = df[add_c].fillna("unknown").astype(str)
    else:
        out["adduct"] = "unknown"

    # 4. polarity
    pol_c = adapter.col("polarity")
    if pol_c and pol_c in df.columns:
        out["polarity"] = df[pol_c].fillna("unknown").astype(str)
    else:
        out["polarity"] = "unknown"

    # 5. ce (collision energy)
    ce_c = adapter.col("collision_energy")
    if ce_c and ce_c in df.columns:
        out["ce"] = pd.to_numeric(
            df[ce_c].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) and len(x) > 0 else x),
            errors="coerce"
        ).astype(np.float32)
    else:
        out["ce"] = np.nan

    # 6. instrument
    inst_c = adapter.col("instrument")
    if inst_c and inst_c in df.columns:
        out["instrument"] = df[inst_c].fillna("unknown").astype(str)
    else:
        out["instrument"] = "unknown"

    # 7. mz & inten - Zero-copy direct series assignment
    mz_c = adapter.col("mz_array")
    int_c = adapter.col("intensity_array")
    out["mz"] = df[mz_c]
    out["inten"] = df[int_c]

    # 8. neutral_mass - Vectorized lookup arithmetic
    is_neg = out["polarity"].str.lower().str.contains("neg")
    default_shift = np.where(is_neg, -PROTON_MASS, PROTON_MASS)
    charges = out["adduct"].map(charge_map).fillna(1.0).to_numpy(dtype=np.float64)
    shifts  = out["adduct"].map(shift_map).fillna(pd.Series(default_shift, index=out.index)).to_numpy(dtype=np.float64)
    out["neutral_mass"] = out["precursor_mz"].to_numpy(dtype=np.float64) * charges - shifts

    # 9. Train-only: smiles and ik14 - Direct assignment from existing columns!
    if is_train:
        smi_c = adapter.col("smiles")
        out["smiles"] = df[smi_c].astype(str)
        ik14_c = adapter.col("inchikey14")
        if ik14_c and ik14_c in df.columns:
            out["ik14"] = df[ik14_c].astype(str)
        elif "inchikey14" in df.columns:
            out["ik14"] = df["inchikey14"].astype(str)
        elif "molecule_id" in df.columns:
            out["ik14"] = df["molecule_id"].astype(str)
        else:
            out["ik14"] = out["molecule_id"]

    # Reorder columns to exact contract
    if is_train:
        cols = ["molecule_id", "smiles", "ik14", "precursor_mz", "adduct", "polarity", "ce", "instrument", "mz", "inten", "neutral_mass"]
    else:
        cols = ["molecule_id", "precursor_mz", "adduct", "polarity", "ce", "instrument", "mz", "inten", "neutral_mass"]
    return out[cols]

t_bld = time.time()
train_clean = build_normalized_parquet_fast(train_raw, adapter_train, is_train=True)
test_clean  = build_normalized_parquet_fast(test_raw, adapter_test, is_train=False)
print(f"Parquet normalization built in {time.time() - t_bld:.3f}s (pure columnar pandas).")

train_clean_path = os.path.join(OUTPUT_DIR, "train_clean.parquet")
test_clean_path  = os.path.join(OUTPUT_DIR, "test_clean.parquet")

train_clean.to_parquet(train_clean_path, index=False)
test_clean.to_parquet(test_clean_path, index=False)

print(f"Saved: {train_clean_path} ({len(train_clean)} rows, {list(train_clean.columns)})")
print(f"Saved: {test_clean_path} ({len(test_clean)} rows, {list(test_clean.columns)})")

del train_clean, test_clean
gc.collect()

In [ ]:
# ==============================================================================
# ARTIFACT SAVING: STATS.JSON & MANIFEST.JSON
# ==============================================================================
check_budget()

def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items() if not k.startswith("_")}
    if isinstance(obj, (list, tuple)):
        return [make_serializable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

stats_payload = {
    "train_spectral": make_serializable(train_spectral_stats),
    "test_spectral": make_serializable(test_spectral_stats),
    "distribution_comparison": make_serializable(distribution_comparison),
    "molecule_stats": make_serializable(molecule_stats),
    "overlap_analysis": make_serializable(overlap_analysis),
    "config": asdict(cfg),
    "timestamp": datetime.now().isoformat(),
}

stats_path = os.path.join(OUTPUT_DIR, "stats.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(stats_payload, f, indent=2)

manifest_payload = {
    "stage": "eda",
    "config": asdict(cfg),
    "library_versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "rdkit": RDKIT_VERSION,
        "scipy": scipy.__version__,
        "torch": TORCH_VERSION,
    },
    "hardware": {
        "n_cpus": os.cpu_count(),
        "n_gpus": GPU_COUNT,
        "devices": gpu_info,
    },
    "files": [
        "schema.json",
        "train_clean.parquet",
        "test_clean.parquet",
        "stats.json",
        "manifest.json",
        "recommendations.md",
        "figures/01_precursor_mz.png",
        "figures/02_peaks_per_spectrum.png",
        "figures/03_adduct_polarity.png",
        "figures/04_mol_properties.png",
        "figures/05_ppm_error.png",
        "figures/06_example_spectra.png",
    ],
    "metrics": {
        "train_rows": train_spectral_stats["n_rows"],
        "test_rows": test_spectral_stats["n_rows"],
        "train_molecules": train_spectral_stats["n_unique_molecules"],
        "test_molecules": test_spectral_stats["n_unique_molecules"],
        "rdkit_parse_rate": molecule_stats["rdkit_parse_rate"],
        "mass_sanity_match_rate": molecule_stats["mass_sanity_check"]["frac_matched"],
        "test_mass_overlap_rate": overlap_analysis["frac_test_mass_covered"],
    },
    "timestamp": datetime.now().isoformat(),
}

manifest_path = os.path.join(OUTPUT_DIR, "manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest_payload, f, indent=2)

print(f"Saved: {stats_path}")
print(f"Saved: {manifest_path}")

## 8. Recommendations (Programmatically Generated)
Synthesizes quantitative spectral and chemical observations into concrete hyperparameter guidelines for subsequent feature-engineering and neural retrieval stages.

In [ ]:
# ==============================================================================
# RECOMMENDATIONS (PROGRAMMATICALLY GENERATED)
# ==============================================================================
from IPython.display import Markdown, display

peak_q95 = train_spectral_stats["peaks_per_spectrum"]["q95"]
peak_q99 = train_spectral_stats["peaks_per_spectrum"]["q99"]
rec_k_peaks = int(min(300, max(60, int(np.ceil(peak_q95 * 1.15)))))

max_mz = train_spectral_stats["precursor_mz"]["max"]
rec_max_mz = int(np.ceil(max_mz + 50.0))

mw_q99 = np.percentile(exact_mws, 99) if len(exact_mws) > 0 else 1200.0
rec_max_mw = int(np.ceil(mw_q99 + 100.0))

rec_lines = [
    "# Modeling & Pipeline Recommendations",
    "",
    "### 1. Spectral Representation & Peak Capping",
    f"* **Recommended Peak Cap (K)**: **`{rec_k_peaks}`** peaks per spectrum.",
    f"  * Captures >95% of spectral peaks (Train 95th percentile = `{peak_q95:.0f}`, 99th percentile = `{peak_q99:.0f}`).",
    f"  * Keeping K <= {rec_k_peaks} bounds GPU memory while preserving key diagnostic fragments.",
    f"* **m/z Bin Range**: `[0, {rec_max_mz}]` Da.",
    "* **Bin Resolution**:",
    f"  * Unit/Low resolution: `0.1 Da` (~{rec_max_mz*10} bins).",
    "  * High resolution / Transformer models: `0.01 Da` or direct continuous embedding (m/z, sqrt(intensity)).",
    "* **Intensity Preprocessing**: Intensities are normalized to base peak = 1.0. Square-root transform is highly recommended to upweight informative low-abundance fragments.",
    "",
    "### 2. Formula & Chemical Search Space Constraints",
    f"* **Maximum Molecular Weight**: **`{rec_max_mw}` Da** (99th percentile train MW = `{mw_q99:.1f}` Da).",
    f"* **Adduct Mass Conversion**: `{molecule_stats['mass_sanity_check']['frac_matched']:.2%}` of train spectra match SMILES exact mass within 10 ppm using the standard adduct table.",
    f"* **Element Universe**: Restrict formula generation to `{list(molecule_stats['element_set'].keys())[:10]}`.",
    "",
    "### 3. Validation & Splitting Strategy",
    "* **Strict Molecule Disjointness**: Multiple spectra exist per molecule. Cross-validation MUST group by `molecule_id` or `ik14`.",
    f"* **Scaffold-Disjoint Split**: Recommended `GroupKFold` on Murcko Scaffold (`{molecule_stats['n_unique_scaffolds']}` unique scaffolds). This accurately mimics novel scaffold generalization required on the test set.",
    "",
    "### 4. Baseline & Retrieval Ceiling",
    f"* **Mass Overlap**: `{overlap_analysis['frac_test_mass_covered']:.2%}` of test molecules share a neutral mass within +/-{cfg.mass_match_ppm} ppm of a training molecule.",
    f"* Pure library mass retrieval has an upper bound MRR@25 of `{overlap_analysis['theoretical_max_library_lookup_mrr']:.2%}`. Structure generation / retrieval over a massive external candidate library is essential.",
]
recommendations_text = "\n".join(rec_lines)

display(Markdown(recommendations_text))

rec_file_path = os.path.join(OUTPUT_DIR, "recommendations.md")
with open(rec_file_path, "w", encoding="utf-8") as f:
    f.write(recommendations_text)
print(f"Saved: {rec_file_path}")

## 9. Final Sanity Checks & Artifact Verification
Asserts that all required artifact files exist on disk, are non-empty, adhere strictly to the column schema, and confirms no data leakage.

In [ ]:
# ==============================================================================
# FINAL SANITY CHECKS & ARTIFACT AUDIT
# ==============================================================================

required_artifacts = [
    "schema.json",
    "train_clean.parquet",
    "test_clean.parquet",
    "stats.json",
    "manifest.json",
    "recommendations.md",
    "figures/01_precursor_mz.png",
    "figures/02_peaks_per_spectrum.png",
    "figures/03_adduct_polarity.png",
    "figures/04_mol_properties.png",
    "figures/05_ppm_error.png",
    "figures/06_example_spectra.png",
]

for art in required_artifacts:
    full_p = os.path.join(OUTPUT_DIR, art)
    assert os.path.exists(full_p), f"Artifact missing: {full_p}"
    assert os.path.getsize(full_p) > 0, f"Artifact empty: {full_p}"

# Validate clean parquets
df_tr_clean = pd.read_parquet(os.path.join(OUTPUT_DIR, "train_clean.parquet"))
df_te_clean = pd.read_parquet(os.path.join(OUTPUT_DIR, "test_clean.parquet"))

expected_tr_cols = {"molecule_id", "smiles", "ik14", "precursor_mz", "adduct",
                    "polarity", "ce", "instrument", "mz", "inten", "neutral_mass"}
expected_te_cols = {"molecule_id", "precursor_mz", "adduct", "polarity",
                    "ce", "instrument", "mz", "inten", "neutral_mass"}

assert expected_tr_cols.issubset(set(df_tr_clean.columns)), f"train_clean missing cols: {expected_tr_cols - set(df_tr_clean.columns)}"
assert expected_te_cols.issubset(set(df_te_clean.columns)), f"test_clean missing cols: {expected_te_cols - set(df_te_clean.columns)}"

# Leakage check: test_clean must NEVER contain smiles or ik14
assert "smiles" not in df_te_clean.columns, "Data Leakage: 'smiles' found in test_clean!"
assert "ik14" not in df_te_clean.columns, "Data Leakage: 'ik14' found in test_clean!"

# Verify schema.json
with open(os.path.join(OUTPUT_DIR, "schema.json"), "r", encoding="utf-8") as f:
    loaded_schema = json.load(f)
assert "train" in loaded_schema and "test" in loaded_schema

del df_tr_clean, df_te_clean
gc.collect()

print("=" * 70)
print("ALL SANITY CHECKS PASSED SUCCESSFULLY [OK]")
print(f"All artifacts verified in: {OUTPUT_DIR}")
print(f"Total notebook execution time: {elapsed():.2f}s")
print("=" * 70)